In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# B-2. 정책 시뮬레이터 (구매수량 모델 + 재고 전방 시뮬레이션 + 정책 평가)
# ============================================================
# 목차
#  1. 준비
#     1-1. 환경 설정 (라이브러리, 경로, 필요 파일 안내)
#     1-2. 데이터·B-1 산출물 로드, 수량모델 학습(1차 vs 고도화 비교)
#     1-3. 공통 상태 준비 (시간격자, 재고, 방문객, memory stream)
#  2. 시뮬레이션 엔진
#     2-1. 정책 적용 규칙 + simulate_policy() 정의
#     2-2. 2단 강도보정 (전역 G 이분탐색 -> hour x category IPF)
#  3. 검증
#     3-1. 기존 정책 재현 vs 실제 (판매·폐기·시간패턴)
#     3-2. 무할인 반사실 (A-1 정합성 검증용)
#     3-3. 후보 정책 비교 데모 + 탄력성 민감도
#  4. 인터페이스 산출물 저장 (A-2·A-3 전달용)
#  5. 결과 요약·다운로드
# ------------------------------------------------------------
# [실행 전 /content에 올릴 파일]
#  csv 8개: customer, product, store, calendar, store_visitor_profile,
#           visitor, receipt, inventory
#  B-1 산출물: b1_models.joblib (+ b1_sim_calibration.json, b1_outputs_final.zip의 models 폴더)
# ------------------------------------------------------------

In [ ]:
# ==== 1-1. 환경 설정 ====
!pip -q install lightgbm catboost 2>/dev/null || true
import numpy as np, pandas as pd, time, json, joblib, warnings, random
from pathlib import Path
warnings.filterwarnings("ignore")
SEED = 42; random.seed(SEED); np.random.seed(SEED)

# B-1 모델 로드에 필요한 클래스 정의 (joblib보다 먼저 실행 필수)
class ElasticityRuleModel:
    def __init__(self, alpha=25.0): self.alpha=alpha
    def fit(self, df, ycol="y"):
        df=df.copy(); df["db"]=(df["discount_rate"]//10).clip(0,4); df["pb"]=(df["price_sensitivity"]*3).astype(int).clip(0,2)
        self.g=df["y"].mean(); self.tbl=df.groupby(["category","db","pb"])["y"].agg(["sum","count"])
        self.cat_m=df.groupby("category")["y"].mean().to_dict(); return self
    def predict_proba(self, df):
        df=df.copy(); df["db"]=(df["discount_rate"]//10).clip(0,4); df["pb"]=(df["price_sensitivity"]*3).astype(int).clip(0,2)
        out=np.zeros(len(df)); a=self.alpha
        for i,(c,d,p) in enumerate(zip(df["category"],df["db"],df["pb"])):
            m=self.cat_m.get(c,self.g)
            try: s,n=self.tbl.loc[(c,d,p)]; out[i]=(s+a*m)/(n+a)
            except KeyError: out[i]=m
        return np.stack([1-out,out],1)

In [ ]:
CAND = [Path("/content"), Path("/mnt/user-data/uploads"), Path(".")]
DATA_DIR = next(d for d in CAND if (d/"customer.csv").exists())
B1_CAND = [Path("/content"), Path("/content/b1_outputs/models"), Path("/home/claude/nb_test_out/models")]
B1_DIR = next(d for d in B1_CAND if (d/"b1_models.joblib").exists())
OUT = Path("./b2_outputs"); (OUT/"interface").mkdir(parents=True, exist_ok=True)
(OUT/"reports").mkdir(exist_ok=True)
print("DATA_DIR:", DATA_DIR, "| B1_DIR:", B1_DIR)

customer_df = pd.read_csv(DATA_DIR/"customer.csv")
product_df = pd.read_csv(DATA_DIR/"product.csv"); product_df["product_id"]=product_df["product_id"].astype(str)
store_df = pd.read_csv(DATA_DIR/"store.csv"); store_df["store_id"]=store_df["store_id"].astype(str)
calendar_df = pd.read_csv(DATA_DIR/"calendar.csv", parse_dates=["date"])
svp_df = pd.read_csv(DATA_DIR/"store_visitor_profile.csv")
visitor_df = pd.read_csv(DATA_DIR/"visitor.csv", parse_dates=["visit_date"])
receipt_df = pd.read_csv(DATA_DIR/"receipt.csv", parse_dates=["sale_date"])
inventory_df = pd.read_csv(DATA_DIR/"inventory.csv", parse_dates=["current_date","expiry_date"])
# [피드백 반영] 데이터 스키마 검증
assert "sold_out_flag" in inventory_df.columns, "구버전 inventory - sold_out_flag 없음"
assert ((inventory_df["inventory_status"]=="SOLD_OUT")==(inventory_df["sold_out_flag"]==1)).all()
assert (inventory_df["inventory_status"]=="EXPIRED").any(), "EXPIRED 행 없음 - 폐기 검증 불가"
print("스키마 검증 통과")
for df in [visitor_df, receipt_df, inventory_df]: df["store_id"]=df["store_id"].astype(str)
for df in [receipt_df, inventory_df]: df["product_id"]=df["product_id"].astype(str)

art = joblib.load(B1_DIR/"b1_models.joblib")
mono, enc = art["mono"], art["enc"]
CATF, NUMF = art["CAT_FEATURES"], art["NUM_FEATURES"]
calp = B1_DIR/"b1_sim_calibration.joblib"
cal = joblib.load(calp) if calp.exists() else json.load(open(B1_DIR/"b1_sim_calibration.json"))
SLOT_F, CAT_F = cal["slot_factor"], cal["cat_factor"]
CONSIDER_SIZE, PREF_BOOST, MEM_DECAY = cal["CONSIDER_SIZE"], cal["PREF_BOOST"], cal["MEM_DECAY"]
print("B-1 로드:", art.get("mono_backend"), "| 프로토콜:", CONSIDER_SIZE, PREF_BOOST, MEM_DECAY)

TRAIN_END = pd.Timestamp("2025-09-30")
SIM_START, SIM_END = pd.Timestamp("2025-12-01"), pd.Timestamp("2025-12-14")

# ---- 시간 격자: hour 추출, hour->slot 매핑 ----
visitor_df["hour"] = visitor_df["visit_time"].str.slice(0,2).astype(int)
hr2slot = {}
for _, r in svp_df.drop_duplicates(["time_slot","start_hour","end_hour"]).iterrows():
    for h in range(int(r["start_hour"]), int(r["end_hour"])): hr2slot[h] = r["time_slot"]
hr2slot[22] = "closing"
HOURS = sorted(visitor_df["hour"].unique())
print("시간 격자:", HOURS[0], "~", HOURS[-1], "| slot 매핑:", {h: hr2slot[h] for h in [10,12,15,18,21]})

# ---- 구매(라벨) 데이터: receipt + 로트 + 고객 ----
rec = receipt_df.merge(inventory_df[["inventory_id","days_to_expiry"]], on="inventory_id", how="left")
rec["discount_rate"] = pd.to_numeric(rec["discount_rate"], errors="coerce").fillna(0).astype(int)
rec = rec.merge(product_df[["product_id","category","base_price","shelf_life_days"]], on="product_id", how="left")
rec = rec.merge(customer_df, on="customer_id", how="left")
rec["disc_flag"] = (rec["discount_rate"]>0).astype(int)
# [피드백 5-2 반영] 1~5 전 범위 사용, 4~5 비중 명시
_sh45 = (rec["quantity"]>=4).mean()
print(f"수량 4~5 비중: {_sh45:.4f} (전 범위 1~5 사용)")
rec["qty_c"] = rec["quantity"].clip(1,5)
rtr = rec[rec["sale_date"] <= TRAIN_END]
rte = rec[rec["sale_date"] > pd.Timestamp("2025-11-15")]
print("수량모델 train:", len(rtr), "test:", len(rte))

# ============ 2. 구매수량 모델: 1차(경험분포) vs 고도화(다항 로짓) ============
# 과대산포 진단
vm = rtr["quantity"].var()/rtr["quantity"].mean()
print(f"분산/평균 = {vm:.3f} -> 양수 조건부 수량이 1~5 저범위 집중 -> Poisson/NB보다 범주형 분포가 단순·적합 (피드백 5-3 표현 정정)")

# 1차: category x disc_flag 경험분포
emp = {}
for (c,f), g in rtr.groupby(["category","disc_flag"]):
    vc = g["qty_c"].value_counts(normalize=True).reindex([1,2,3,4,5], fill_value=1e-9)
    emp[(c,f)] = vc.to_numpy()
def emp_proba(df):
    return np.stack([emp.get((c,f), np.array([.7,.2,.08,.015,.005])) for c,f in zip(df["category"], df["disc_flag"])])

# 고도화: 다항 로짓 (가구원수 등 고객특성 반영)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
QCAT = ["category","household_type","age_group","income_level"]
QNUM = ["disc_flag","price_sensitivity","base_price"]
qty_pipe = Pipeline([("prep", ColumnTransformer([
    ("c", OneHotEncoder(handle_unknown="ignore"), QCAT),
    ("n", StandardScaler(), QNUM)])),
    ("clf", LogisticRegression(max_iter=1000))])
t0=time.time(); qty_pipe.fit(rtr[QCAT+QNUM], rtr["qty_c"]); print("qty model fitted", round(time.time()-t0,1),"s")

p_emp = emp_proba(rte); p_mnl = qty_pipe.predict_proba(rte[QCAT+QNUM])
ll_emp = log_loss(rte["qty_c"], p_emp, labels=[1,2,3,4,5])
ll_mnl = log_loss(rte["qty_c"], p_mnl, labels=[1,2,3,4,5])
em_emp = (p_emp*np.array([1,2,3,4,5])).sum(1); em_mnl=(p_mnl*np.array([1,2,3,4,5])).sum(1)
cmp_q = pd.DataFrame({"실제평균": rte.groupby("household_type")["qty_c"].mean(),
                      "경험분포": pd.Series(em_emp, index=rte.index).groupby(rte["household_type"]).mean(),
                      "다항로짓": pd.Series(em_mnl, index=rte.index).groupby(rte["household_type"]).mean()})
print(f"\n[수량모델 비교] test LogLoss: 경험분포 {ll_emp:.4f} vs 다항로짓 {ll_mnl:.4f}")
print("가구유형별 평균 구매수량 재현:"); print(cmp_q.round(3).to_string())
QTY_BACKEND = "다항로짓" if ll_mnl < ll_emp else "경험분포"
print("수량모델 채택:", QTY_BACKEND)
# 시뮬레이터용 고속 lookup: (category, disc, household, age, income, ps중앙값, base_price) 조합 격자
grid = []
ps_med = customer_df["price_sensitivity"].median()
for c in product_df["category"].unique():
    bp = product_df[product_df["category"]==c]["base_price"].median()
    for f in [0,1]:
        for h in customer_df["household_type"].unique():
            for a in customer_df["age_group"].unique():
                for i in customer_df["income_level"].unique():
                    grid.append((c,f,h,a,i,ps_med,bp))
G = pd.DataFrame(grid, columns=["category","disc_flag","household_type","age_group","income_level","price_sensitivity","base_price"])
GP = qty_pipe.predict_proba(G[QCAT+QNUM])
qty_lut = {tuple(r[:5]): GP[i] for i,r in enumerate(grid)}
print("qty lookup:", len(qty_lut), "조합")

# ============ 4. 정책 테이블 (A-2 인터페이스) ============
# 스키마: store_id, product_id, date, hour(-1=전시간), dte_min, dte_max, discount_rate(0~40 정수%)
def existing_policy(start, end):
    m = inventory_df[(inventory_df.current_date>=start)&(inventory_df.current_date<=end)&(inventory_df.discount_rate>0)]
    p = (m.groupby(["store_id","product_id","current_date","days_to_expiry"], as_index=False)
          .agg(discount_rate=("discount_rate","max")).rename(columns={"current_date":"date"}))
    p["dte_min"]=p["days_to_expiry"]; p["dte_max"]=p["days_to_expiry"]
    # [피드백 6-1 반영] 할인 시작 시각 복원: 그날 해당 상품의 첫 할인 판매 시각부터 적용
    rc = receipt_df[(receipt_df.sale_date>=start)&(receipt_df.sale_date<=end)]
    rc = rc[pd.to_numeric(rc["discount_rate"],errors="coerce").fillna(0)>0].copy()
    rc["h0"] = rc["sale_time"].str.slice(0,2).astype(int)
    sh = rc.groupby(["store_id","product_id","sale_date"])["h0"].min().rename("start_h").reset_index()
    sh = sh.rename(columns={"sale_date":"date"})
    p = p.merge(sh, on=["store_id","product_id","date"], how="left")
    p["start_h"] = p["start_h"].fillna(10).astype(int)   # 관측 없으면 개점부터(전시간과 동일)
    rows=[]
    for r in p.itertuples():
        for h in range(int(r.start_h), 23):
            rows.append((r.store_id, r.product_id, r.date, h, r.dte_min, r.dte_max, r.discount_rate))
    return pd.DataFrame(rows, columns=["store_id","product_id","date","hour","dte_min","dte_max","discount_rate"])
def uniform_policy(start, end, rate, dte_max=99):
    days = pd.date_range(start,end); rows=[]
    for s in store_df["store_id"]:
        for pid in product_df["product_id"]:
            for d in days: rows.append((s,pid,d,-1,0,dte_max,rate))
    return pd.DataFrame(rows, columns=["store_id","product_id","date","hour","dte_min","dte_max","discount_rate"])
def closing_boost_policy(start, end):
    """데모 후보: 임박(D-1 이하) 로트를 18시 이전 20%, 18시부터 40% 할인."""
    days = pd.date_range(start,end); rows=[]
    for s in store_df["store_id"]:
        for pid in product_df["product_id"]:
            for d in days:
                for h in range(10,18): rows.append((s,pid,d,h,0,1,20))
                for h in range(18,23): rows.append((s,pid,d,h,0,1,40))
    return pd.DataFrame(rows, columns=["store_id","product_id","date","hour","dte_min","dte_max","discount_rate"])
pol_exist = existing_policy(SIM_START, SIM_END)
print("\n기존 정책 테이블:", pol_exist.shape)

# ============ 5. 탄력성 보정 훅 ============
# 조건부 경험 탄력성: 임박(dte<=1) 구간에서 카테고리 고정, 고할인(>=30) vs 무할인 y-rate 비 (판매라인/오퍼노출 근사)
te_all = rec[(rec["sale_date"]>pd.Timestamp("2025-11-15"))]
near = te_all[te_all["days_to_expiry"]<=1]
lift_by_cat = []
for c,g in near.groupby("category"):
    n0 = (g["discount_rate"]==0).sum(); nH=(g["discount_rate"]>=30).sum()
    if n0>30 and nH>30:
        # 노출 대비 아닌 판매건 비중 비교의 대리: 동일 dte에서 고할인 판매강도/무할인 판매강도
        lift_by_cat.append(nH/max(n0,1))
EMP_LIFT = 0.18   # B-1 5-3에서 산출한 조건부 상승폭(무할인 대비 40% 할인 시)
LR_LIFT = 0.28    # LR 모델 상한
def calib_k(target_lift, sample_n=3000):
    """odds 배율 g(d)=exp(k*d) 로 0->40% 평균확률 상승이 target이 되도록 k 이분탐색."""
    te_s = near.sample(min(sample_n,len(near)), random_state=SEED)
    # mono 모델 기반 기준확률: near-expiry 문맥 평균 반응(0.0465->0.0470, B-1 실측) 사용해 해석적 근사
    p0 = 0.0465; p40_model = 0.0470
    lo, hi = 0.0, 12.0
    for _ in range(60):
        k = (lo+hi)/2
        g = np.exp(k*0.40)
        p40 = g*p40_model/(g*p40_model+(1-p40_model))
        (lo, hi) = (k, hi) if p40/p0 < 1+target_lift else (lo, k)
    return round((lo+hi)/2, 4)
K_EMP, K_LR = calib_k(EMP_LIFT), calib_k(LR_LIFT)
# [피드백 4-1 반영] 라벨 정정: 인과 추정이 아닌 탄력성 "가정" 시나리오
ELASTICITY = {"off": 0.0, "mid": K_EMP, "upper": K_LR}   # mid=중간 탄력성 가정(+18%), upper=상단 가정(+28%)
print("탄력성 훅 k:", ELASTICITY, "(g(d)=exp(k·d), odds scaling)")

joblib.dump(dict(qty_pipe=qty_pipe, qty_lut=qty_lut, emp=emp, QTY_BACKEND=QTY_BACKEND,
                 hr2slot=hr2slot, ELASTICITY=ELASTICITY), OUT/"b2_core.joblib")
pol_exist.to_pickle(OUT/"pol_exist.pkl")
print("core saved ->", OUT)

In [ ]:
CAND = [Path("/content"), Path("/mnt/user-data/uploads"), Path(".")]
DATA_DIR = next(d for d in CAND if (d/"customer.csv").exists())
B1_CAND = [Path("/content"), Path("/content/b1_outputs/models"), Path("/home/claude/nb_test_out/models")]
B1_DIR = next(d for d in B1_CAND if (d/"b1_models.joblib").exists())
OUT = Path("./b2_outputs")

customer_df = pd.read_csv(DATA_DIR/"customer.csv")
product_df = pd.read_csv(DATA_DIR/"product.csv"); product_df["product_id"]=product_df["product_id"].astype(str)
store_df = pd.read_csv(DATA_DIR/"store.csv"); store_df["store_id"]=store_df["store_id"].astype(str)
calendar_df = pd.read_csv(DATA_DIR/"calendar.csv", parse_dates=["date"])
visitor_df = pd.read_csv(DATA_DIR/"visitor.csv", parse_dates=["visit_date"])
receipt_df = pd.read_csv(DATA_DIR/"receipt.csv", parse_dates=["sale_date"])
inventory_df = pd.read_csv(DATA_DIR/"inventory.csv", parse_dates=["current_date"])
for df in [visitor_df, receipt_df, inventory_df]: df["store_id"]=df["store_id"].astype(str)
for df in [receipt_df, inventory_df]: df["product_id"]=df["product_id"].astype(str)

core = joblib.load(OUT/"b2_core.joblib")
emp_raw, hr2slot, ELASTICITY = core["emp"], core["hr2slot"], core["ELASTICITY"]
art = joblib.load(B1_DIR/"b1_models.joblib")
mono, enc = art["mono"], art["enc"]
CATF, NUMF = art["CAT_FEATURES"], art["NUM_FEATURES"]
calp = B1_DIR/"b1_sim_calibration.joblib"
cal = joblib.load(calp) if calp.exists() else json.load(open(B1_DIR/"b1_sim_calibration.json"))
SLOT_F, CAT_F = cal["slot_factor"], cal["cat_factor"]
CONSIDER_SIZE, PREF_BOOST, MEM_DECAY = cal["CONSIDER_SIZE"], cal["PREF_BOOST"], cal["MEM_DECAY"]

TRAIN_END = pd.Timestamp("2025-09-30")
SIM_START, SIM_END = pd.Timestamp("2025-12-01"), pd.Timestamp("2025-12-14")
visitor_df["hour"] = visitor_df["visit_time"].str.slice(0,2).astype(int)
HOURS = sorted(visitor_df["hour"].unique())
stores = sorted(store_df["store_id"])

# 인덱스 맵
cust_ids = customer_df["customer_id"].to_numpy(); cust_i = {c:i for i,c in enumerate(cust_ids)}
cats = sorted(product_df["category"].unique()); cat_i = {c:i for i,c in enumerate(cats)}
prod_ids = product_df["product_id"].to_numpy(); prod_i = {p:i for i,p in enumerate(prod_ids)}
nC, nK, nP = len(cust_ids), len(cats), len(prod_ids)
prod_cat = product_df.set_index("product_id").loc[prod_ids, "category"].to_numpy()
prod_price = product_df.set_index("product_id").loc[prod_ids, "base_price"].to_numpy(float)
prod_shelf = product_df.set_index("product_id").loc[prod_ids, "shelf_life_days"].to_numpy(float)
prod_decay = product_df.set_index("product_id").loc[prod_ids, "freshness_decay_type"].astype(str).to_numpy()
pref_arr = customer_df["preferred_category"].to_numpy()
hh_arr = customer_df["household_type"].astype(str).to_numpy()
age_arr = customer_df["age_group"].astype(str).to_numpy()
inc_arr = customer_df["income_level"].astype(str).to_numpy()
cust_feat = customer_df.set_index("customer_id")
store_feat = store_df.set_index("store_id")[["area_type","floating_idx"]]
cal_feat = calendar_df.set_index("date")

# 신선도 근사: freshness ~ a*(dte/shelf)+b (inventory에서 적합)
inv_s = inventory_df.sample(min(30000, len(inventory_df)), random_state=42).merge(
    product_df[["product_id","shelf_life_days"]], on="product_id")
ratio = (inv_s["days_to_expiry"]/inv_s["shelf_life_days"]).clip(0,1)
A_f, B_f = np.polyfit(ratio, inv_s["freshness_score"], 1)
def fresh_of(dte, pi):
    return np.clip(A_f*np.clip(dte/prod_shelf[pi],0,1)+B_f, 0, 1)

# 임의 시점 재고 초기화 / 입고 이벤트 (정책 시뮬 기간마다 호출)
def build_stock(start, end):
    # [피드백 3-2 반영] 판매 자격: 미품절 & 비만료 & 잔여기한>=0 & 가용>0
    elig = ((inventory_df["sold_out_flag"]==0) & (inventory_df["inventory_status"]!="EXPIRED")
            & (inventory_df["days_to_expiry"]>=0) & (inventory_df["available_qty"]>0))
    snap = inventory_df[elig & (inventory_df.current_date==start)]
    init_lots = {s: [[],[],[],[],[]] for s in stores}
    for r in snap.itertuples():
        L = init_lots[r.store_id]
        L[0].append(prod_i[r.product_id]); L[1].append(int(r.days_to_expiry))
        L[2].append(int(r.available_qty)); L[3].append(float(r.unit_cost)); L[4].append(float(r.unit_price))
    inb = inventory_df[(inventory_df.current_date>start)&(inventory_df.current_date<=end)&(inventory_df.inbound_qty>0)&(inventory_df.days_to_expiry>=0)]
    inbound = {}
    for r in inb.itertuples():
        inbound.setdefault((r.store_id, r.current_date), []).append(
            (prod_i[r.product_id], int(r.days_to_expiry), int(r.inbound_qty), float(r.unit_cost), float(r.unit_price)))
    return init_lots, inbound

def visit_counts(start, end):
    vc = visitor_df[(visitor_df.visit_date>=start)&(visitor_df.visit_date<=end)]
    return vc.groupby(["store_id","visit_date","hour"]).size().to_dict()

w = visitor_df[visitor_df.visit_date<=TRAIN_END].groupby("customer_id").size()
cust_w = np.full(nC, 0.1)
for cid, v in w.items(): cust_w[cust_i[cid]] = v+0.1
cust_p = cust_w/cust_w.sum()

# 수량 경험분포 -> 샘플링 형태
emp_s = {k: (np.array([1,2,3,4,5]), v/v.sum()) for k, v in emp_raw.items()}
qty_lut = core.get("qty_lut", {}); QTY_BACKEND = core.get("QTY_BACKEND","경험분포")
def sample_qty(rng, cat, disc_flag, cu):
    """[피드백 5-1 반영] 채택 백엔드로 분기해 수량 샘플"""
    if QTY_BACKEND != "경험분포":
        pr = qty_lut.get((cat, disc_flag, hh_arr[cu], age_arr[cu], inc_arr[cu]))
        if pr is not None:
            return int(rng.choice(np.arange(1, len(pr)+1), p=pr/pr.sum()))
    vals, probs = emp_s.get((cat, disc_flag), (np.array([1]), np.array([1.0])))
    return int(rng.choice(vals, p=probs))

# memory stream 재구축 (receipt에서)
events = receipt_df.merge(product_df[["product_id","category"]], on="product_id")[
    ["customer_id","sale_date","category","product_id","discount_rate"]].copy()
events["discount_rate"] = pd.to_numeric(events["discount_rate"], errors="coerce").fillna(0)
def build_decay_state(ev, keys):
    ev = ev.sort_values(keys+["sale_date"]).reset_index(drop=True)
    grp = ev.groupby(keys, sort=False).ngroup().to_numpy()
    dates = ev["sale_date"].values.astype("datetime64[D]").astype(int)
    sc = np.zeros(len(ev)); sp, dp, gp = 0.0, 0, -1
    for i in range(len(ev)):
        s = 1.0 if grp[i]!=gp else sp*(MEM_DECAY**(dates[i]-dp))+1.0
        sc[i]=s; sp,dp,gp = s,dates[i],grp[i]
    ev["mem_score_at_event"]=sc; return ev
ev_cat = build_decay_state(events, ["customer_id","category"])
ev_prod = build_decay_state(events, ["customer_id","product_id"])
ev_cust = events.sort_values(["customer_id","sale_date"]).reset_index(drop=True)
gc = ev_cust.groupby("customer_id", sort=False).ngroup().to_numpy()
dr = ev_cust["discount_rate"].to_numpy(float); ew=np.zeros(len(ev_cust)); prev=0.0; gp=-1
for i in range(len(ev_cust)):
    prev = dr[i] if gc[i]!=gp else 0.8*prev+0.2*dr[i]
    ew[i]=prev; gp=gc[i]
ev_cust["deal_ewma_at_event"]=ew

def init_all_mem(asof):
    def snap_(ev, keys, shape, maps):
        sc = np.zeros(shape); la = np.full(shape, -10**6, dtype=np.int64)
        e = ev[ev["sale_date"]<asof].sort_values("sale_date").groupby(keys, as_index=False).last()
        di = e["sale_date"].values.astype("datetime64[D]").astype(np.int64)
        for r in range(len(e)):
            ii = tuple(m[e[k].iloc[r]] for m,k in zip(maps,keys))
            sc[ii]=e["mem_score_at_event"].iloc[r]; la[ii]=di[r]
        return sc, la
    a,b = snap_(ev_cat, ["customer_id","category"], (nC,nK), [cust_i,cat_i])
    c,d = snap_(ev_prod, ["customer_id","product_id"], (nC,nP), [cust_i,prod_i])
    e = ev_cust[ev_cust["sale_date"]<asof].groupby("customer_id").last()
    de = np.full(nC,-1.0)
    for cid,v in e["deal_ewma_at_event"].items(): de[cust_i[cid]]=v
    return a,b,c,d,de

def predict_prob(F):
    X = np.hstack([enc.transform(F[CATF]), F[NUMF].to_numpy(float)])
    return mono.predict_proba(X)[:,1]

def assemble_features(sh, of, o_pi, o_dc, o_dte, o_qty, o_cat, slot, cal, sid,
                      mc_s, mc_l, mp_s, mp_l, deal, d_int):
    pi = o_pi[of]; dc = o_dc[of]; dte = o_dte[of]
    n = len(sh)
    # 오퍼 문맥: 같은 상품의 변형 수 / 정가 신선 대안
    vc_ = {}
    fa_ = {}
    for p_, d_, t_ in zip(o_pi, o_dc, o_dte):
        vc_[p_] = vc_.get(p_,0)+1
        if d_==0 and t_>2: fa_[p_]=1
    F = pd.DataFrame({
        "category": o_cat[of], "base_price": prod_price[pi], "shelf_life_days": prod_shelf[pi],
        "freshness_decay_type": prod_decay[pi], "discount_rate": dc, "discount_ratio": dc/100.0,
        "days_to_expiry": dte, "freshness_score": fresh_of(dte, pi)})
    C = cust_feat.iloc[sh].reset_index()
    F = pd.concat([F, C], axis=1)
    F["time_slot"]=slot; F["day_type_cal"]=cal["day_type"]; F["season"]=cal["season"]
    F["day_of_week"]=cal["day_of_week"]; F["is_weekend"]=cal["is_weekend"]
    F["is_holiday"]=cal["is_holiday"]; F["month"]=cal["month"]
    F["area_type"]=store_feat.loc[sid,"area_type"]; F["floating_idx"]=store_feat.loc[sid,"floating_idx"]
    ci_arr = np.array([cat_i[c] for c in F["category"]])
    gc_ = d_int - mc_l[sh, ci_arr]; gp_ = d_int - mp_l[sh, pi]
    F["mem_cat_score"]=mc_s[sh,ci_arr]*(MEM_DECAY**np.clip(gc_,0,500))
    F["mem_cat_days_since"]=np.clip(gc_,0,999)
    F["mem_prod_score"]=mp_s[sh,pi]*(MEM_DECAY**np.clip(gp_,0,500))
    F["mem_prod_days_since"]=np.clip(gp_,0,999)
    F["mem_cust_deal_ewma"]=deal[sh]
    F["is_pref_category"]=(F["category"]==F["preferred_category"]).astype(int)
    F["expiry_ratio"]=(F["days_to_expiry"]/F["shelf_life_days"]).clip(0,1)
    F["log_base_price"]=np.log1p(F["base_price"])
    F["ps_x_disc"]=F["price_sensitivity"]*F["discount_ratio"]
    F["fs_x_fresh"]=F["freshness_sensitivity"]*F["freshness_score"]
    F["variant_count"]=[vc_.get(p_,1) for p_ in pi]
    F["fresh_alt_available"]=[fa_.get(p_,0) for p_ in pi]
    F["near_expiry"]=(F["days_to_expiry"]<=1).astype(int)
    F["disc_x_near"]=F["discount_ratio"]*F["near_expiry"]
    for c in CATF: F[c]=F[c].astype(str)
    return F


# 정책 생성기 (core에서 복제)
def existing_policy(start, end):
    m = inventory_df[(inventory_df.current_date>=start)&(inventory_df.current_date<=end)&(inventory_df.discount_rate>0)]
    p = (m.groupby(["store_id","product_id","current_date","days_to_expiry"], as_index=False)
          .agg(discount_rate=("discount_rate","max")).rename(columns={"current_date":"date"}))
    p["hour"]=-1; p["dte_min"]=p["days_to_expiry"]; p["dte_max"]=p["days_to_expiry"]
    return p[["store_id","product_id","date","hour","dte_min","dte_max","discount_rate"]]
def uniform_policy(start, end, rate, dte_max=99):
    days = pd.date_range(start,end); rows=[]
    for s in stores:
        for pid in prod_ids:
            for d in days: rows.append((s,pid,d,-1,0,dte_max,rate))
    return pd.DataFrame(rows, columns=["store_id","product_id","date","hour","dte_min","dte_max","discount_rate"])
def closing_boost_policy(start, end):
    days = pd.date_range(start,end); rows=[]
    for s in stores:
        for pid in prod_ids:
            for d in days:
                for h in range(10,18): rows.append((s,pid,d,h,0,1,20))
                for h in range(18,23): rows.append((s,pid,d,h,0,1,40))
    return pd.DataFrame(rows, columns=["store_id","product_id","date","hour","dte_min","dte_max","discount_rate"])
pol_exist = existing_policy(SIM_START, SIM_END)

In [ ]:
# ==== 2-1. 정책 적용 규칙 + 시뮬레이션 엔진 ====
def lot_discounts(prows, prod_arr, dte_arr, hour):
    disc = np.zeros(len(prod_arr), dtype=int)
    for i in range(len(prod_arr)):
        best = (-1, -1)
        for (h, dmin, dmax, r) in prows.get(prod_arr[i], ()):
            if (h == -1 or h == hour) and dmin <= dte_arr[i] <= dmax:
                pr = (2 if h == hour else 0) + (1 if dmax - dmin < 99 else 0)
                if pr > best[0] or (pr == best[0] and r > best[1]):
                    best = (pr, r)
        if best[1] > 0: disc[i] = best[1]
    return disc

def simulate_policy(policy_df, start, end, n_runs=5, elasticity="off",
                    hour_factor=None, cat_factor=None, seed0=9000, collect_detail=False,
                    anchor_scale=1.0):
    """anchor_scale: A-1 수요 시나리오 배율 (P10/P50/P90). 확률은 '후보 오퍼 내 선택확률'."""
    """정책 테이블 -> 재고 전방 몬테카를로. 반환: (요약, sales, waste)."""
    k = ELASTICITY[elasticity]
    HF = hour_factor or {}; CF = cat_factor or {}
    pol = policy_df.copy()
    if len(pol): pol["date"] = pd.to_datetime(pol["date"])
    pol_g = {}
    for r in pol.itertuples():
        pol_g.setdefault((r.store_id, r.date), {}).setdefault(r.product_id, []).append(
            (int(r.hour), int(r.dte_min), int(r.dte_max), int(r.discount_rate)))
    days = pd.date_range(start, end)
    init_lots, inbound = build_stock(start, end)
    vis_cnt = visit_counts(start, end)
    m0 = init_all_mem(start)
    summaries, det_sales, det_waste = [], [], []
    for run in range(n_runs):
        rng = np.random.default_rng(SEED + seed0 + run)
        mc_s, mc_l, mp_s, mp_l, deal = [x.copy() for x in m0]
        lots = {s: [list(a) for a in init_lots[s]] for s in stores}
        S = dict(qty=0, rev=0.0, gross=0.0, margin=0.0, wq=0, wc=0.0)
        for d in days:
            d_int = np.int64(d.to_datetime64().astype("datetime64[D]").astype(np.int64))
            cal = cal_feat.loc[d]
            for s in stores:
                for lot in inbound.get((s, d), ()):
                    for arr, v in zip(lots[s], lot): arr.append(v)
                prows = pol_g.get((s, d), {})
                for h in HOURS:
                    n_v = vis_cnt.get((s, d, h), 0)
                    if n_v == 0: continue
                    prod_l = np.array(lots[s][0], dtype=int); dte_l = np.array(lots[s][1])
                    qty_l = np.array(lots[s][2], dtype=float)
                    if not (qty_l > 0).any(): continue
                    disc_l = lot_discounts(prows, prod_ids[prod_l], dte_l, h)
                    key = prod_l * 100 + disc_l
                    off_map = {}
                    for i in np.where(qty_l > 0)[0]:
                        off_map.setdefault(key[i], []).append(i)
                    okeys = list(off_map)
                    o_pi = np.array([kk // 100 for kk in okeys]); o_dc = np.array([kk % 100 for kk in okeys])
                    o_dte = np.array([dte_l[off_map[kk]].min() for kk in okeys])
                    o_qty = np.array([qty_l[off_map[kk]].sum() for kk in okeys])
                    o_cat = prod_cat[o_pi]
                    slot = hr2slot[h]
                    shoppers = rng.choice(nC, size=min(int(n_v), nC), replace=False, p=cust_p)
                    nS, nO = len(shoppers), len(okeys)
                    # [B-1 v3 규칙 통일] 재고 가중 노출 + 비복원 추출
                    p_exp = (o_qty.astype(float).clip(0) + 1e-9); p_exp = p_exp/p_exp.sum()
                    kk = min(CONSIDER_SIZE, nO)
                    idx = np.stack([rng.choice(nO, size=kk, replace=False, p=p_exp) for _ in range(nS)])
                    use_pref = rng.random((nS, kk)) < PREF_BOOST
                    prefv = pref_arr[shoppers]
                    cat_idx = {c: np.where(o_cat == c)[0] for c in np.unique(o_cat)}
                    for j in range(nS):
                        ci = cat_idx.get(prefv[j])
                        if ci is None: continue
                        tk = use_pref[j]
                        if tk.any():
                            pc = p_exp[ci]/p_exp[ci].sum()
                            picks = rng.choice(ci, size=min(tk.sum(), len(ci)), replace=False, p=pc)
                            slots = np.where(tk)[0][:len(picks)]
                            for s_, pk in zip(slots, picks):
                                if pk not in idx[j]: idx[j, s_] = pk
                    sh = np.repeat(shoppers, CONSIDER_SIZE); of = idx.ravel()
                    F = assemble_features(sh, of, o_pi, o_dc, o_dte, o_qty, o_cat, slot, cal, s,
                                          mc_s, mc_l, mp_s, mp_l, deal, d_int)
                    p = predict_prob(F)
                    dr = o_dc[of] / 100.0
                    f = anchor_scale * HF.get(h, 1.0) * np.array([CF.get(c, 1.0) for c in o_cat[of]]) * np.exp(k * dr)
                    p = np.clip(f * p / (f * p + (1 - p)), 0, 0.98)
                    buy = rng.random(len(p)) < p
                    for jj in rng.permutation(np.where(buy)[0]):
                        ok = okeys[of[jj]]
                        rows = [i for i in off_map[ok] if qty_l[i] > 0]
                        if not rows: continue
                        c_ = o_cat[of[jj]]; fdc = int(o_dc[of[jj]] > 0)
                        q = sample_qty(rng, c_, fdc, int(sh[jj]))
                        pi_ = int(o_pi[of[jj]]); price0 = float(prod_price[pi_])
                        price = round(price0 * (1 - o_dc[of[jj]] / 100.0))
                        sold = 0
                        for i in sorted(rows, key=lambda i: dte_l[i]):
                            take = int(min(q - sold, qty_l[i]))
                            qty_l[i] -= take; lots[s][2][i] = qty_l[i]; sold += take
                            S["margin"] += take * (price - lots[s][3][i])
                            if sold >= q: break
                        if sold == 0: continue
                        S["qty"] += sold; S["rev"] += price * sold; S["gross"] += price0 * sold
                        if collect_detail:
                            det_sales.append((run, s, d, h, prod_ids[pi_], int(o_dc[of[jj]]), sold, price * sold))
                        cu = sh[jj]; ci_ = cat_i[c_]
                        g_ = d_int - mc_l[cu, ci_]
                        mc_s[cu, ci_] = mc_s[cu, ci_] * (MEM_DECAY ** max(g_, 0)) + 1.0; mc_l[cu, ci_] = d_int
                        g_ = d_int - mp_l[cu, pi_]
                        mp_s[cu, pi_] = mp_s[cu, pi_] * (MEM_DECAY ** max(g_, 0)) + 1.0; mp_l[cu, pi_] = d_int
                        deal[cu] = o_dc[of[jj]] if deal[cu] < 0 else 0.8 * deal[cu] + 0.2 * o_dc[of[jj]]
                np_, nd_, nq_, nc_, npr_ = [], [], [], [], []
                for i in range(len(lots[s][0])):
                    q = lots[s][2][i]
                    if lots[s][1][i] <= 0:
                        if q > 0:
                            S["wq"] += q; S["wc"] += q * lots[s][3][i]
                            if collect_detail:
                                det_waste.append((run, s, d, prod_ids[int(lots[s][0][i])], q, q * lots[s][3][i]))
                    elif q > 0:
                        np_.append(lots[s][0][i]); nd_.append(lots[s][1][i] - 1); nq_.append(q)
                        nc_.append(lots[s][3][i]); npr_.append(lots[s][4][i])
                lots[s] = [np_, nd_, nq_, nc_, npr_]
        S["profit"] = S["margin"] - S["wc"]; S["run"] = run
        summaries.append(S)
    out = pd.DataFrame(summaries)
    sales_df = pd.DataFrame(det_sales, columns=["run","store_id","date","hour","product_id","discount_rate","qty","revenue"]) if collect_detail else None
    waste_df = pd.DataFrame(det_waste, columns=["run","store_id","date","product_id","waste_qty","waste_cost"]) if collect_detail else None
    return out, sales_df, waste_df



In [ ]:
# ============ 6-3. B-2 자체 강도보정: (1) 전역 배율 이분탐색 -> (2) hour x cat IPF ============
CAL_START, CAL_END = pd.Timestamp("2025-10-15"), pd.Timestamp("2025-10-28")
pol_cal = existing_policy(CAL_START, CAL_END)
ca = receipt_df[(receipt_df.sale_date>=CAL_START)&(receipt_df.sale_date<=CAL_END)].copy()
ca["hour"] = ca["sale_time"].str.slice(0,2).astype(int)
ca = ca.merge(product_df[["product_id","category"]], on="product_id")
A_tot = ca["quantity"].sum()
t0 = time.time()
lo, hi = 0.2, 1.5
for it in range(5):
    G = (lo+hi)/2
    sm,_,_ = simulate_policy(pol_cal, CAL_START, CAL_END, n_runs=1, elasticity="off",
                             hour_factor={h:G for h in HOURS}, seed0=500)
    q = sm["qty"].iloc[0]
    print(f"  bisect it{it}: G={G:.3f} qty={q:.0f} (target {A_tot})")
    lo, hi = (G, hi) if q < A_tot else (lo, G)
G = (lo+hi)/2
_, cs, _ = simulate_policy(pol_cal, CAL_START, CAL_END, n_runs=2, elasticity="off",
                           hour_factor={h:G for h in HOURS}, seed0=600, collect_detail=True)
cs = cs.merge(product_df[["product_id","category"]], on="product_id")
M_a = ca.groupby(["hour","category"])["quantity"].sum().unstack(fill_value=0.0)
M_s = (cs.groupby(["run","hour","category"])["qty"].sum().groupby(["hour","category"]).mean()
       .unstack(fill_value=0.0).reindex(index=M_a.index, columns=M_a.columns).fillna(1e-9))
hf = pd.Series(1.0, index=M_a.index); cf = pd.Series(1.0, index=M_a.columns)
for _ in range(3):
    hf = hf * (M_a.sum(1) / (M_s.mul(cf, axis=1).mul(hf, axis=0)).sum(1))
    cf = cf * (M_a.sum(0) / (M_s.mul(cf, axis=1).mul(hf, axis=0)).sum(0))
hf = hf.clip(0.25, 4.0); cf = cf.clip(0.4, 2.5)
hf = hf/np.exp(np.log(hf).mean()); cf = cf/np.exp(np.log(cf).mean())   # 기하평균 1로 정규화(총량은 G가 담당)
HOUR_F = {h: float(G*hf.get(h,1.0)) for h in HOURS}; CAT_F2 = {c: float(v) for c,v in cf.items()}
print(f"B-2 2단 보정 {time.time()-t0:.0f}s | G={G:.3f} | hour(대표):",
      {h: round(HOUR_F[h],2) for h in [10,12,15,18,21,22]}, "| cat:", {c: round(v,2) for c,v in CAT_F2.items()})
fac_G = float(G)
joblib.dump(dict(HOUR_F=HOUR_F, CAT_F2=CAT_F2, G=fac_G), OUT/"b2_factors.joblib")

In [ ]:
# ==== 3-1. 기존 정책 재현 검증 ====
# 7-1 기존정책 검증(10 runs) + 상태 저장
t0=time.time()
sum_ex, sal_ex, was_ex = simulate_policy(pol_exist, SIM_START, SIM_END, n_runs=10, elasticity="off",
                                         hour_factor=HOUR_F, cat_factor=CAT_F2, collect_detail=True)
print(f"7-1 {time.time()-t0:.0f}s")
act = receipt_df[(receipt_df.sale_date>=SIM_START)&(receipt_df.sale_date<=SIM_END)]
act_w = inventory_df[(inventory_df.current_date>=SIM_START)&(inventory_df.current_date<=SIM_END)]["daily_waste_qty"].sum()
res = dict(act_qty=int(act['quantity'].sum()), act_rev=float(act['line_amount'].sum()), act_waste=int(act_w),
           act_disc=float(act.loc[act.discount_rate>0,'quantity'].sum()/act['quantity'].sum()),
           sim_qty=float(sum_ex['qty'].mean()), sim_qty_sd=float(sum_ex['qty'].std()),
           sim_rev=float(sum_ex['rev'].mean()), sim_waste=float(sum_ex['wq'].mean()), sim_waste_sd=float(sum_ex['wq'].std()),
           sim_disc=float(sal_ex.loc[sal_ex.discount_rate>0,'qty'].sum()/sal_ex['qty'].sum()),
           sim_profit=float(sum_ex['profit'].mean()))
a_h = act.assign(hour=act["sale_time"].str.slice(0,2).astype(int)).groupby("hour")["quantity"].sum()
s_h = sal_ex.groupby(["run","hour"])["qty"].sum().groupby("hour").mean()
res["hour_corr"]=float(np.corrcoef(a_h.to_numpy(), s_h.reindex(a_h.index).fillna(0).to_numpy())[0,1])
a_d = act.groupby("sale_date")["quantity"].sum(); s_d = sal_ex.groupby(["run","date"])["qty"].sum().groupby("date").mean()
res["day_corr"]=float(np.corrcoef(a_d.to_numpy(), s_d.reindex(a_d.index).to_numpy())[0,1])
joblib.dump(res, OUT/"s_val.joblib")
sum_ex.to_pickle(OUT/"s_sum_ex.pkl"); sal_ex.to_pickle(OUT/"s_sal_ex.pkl"); was_ex.to_pickle(OUT/"s_was_ex.pkl")
print(res)

In [ ]:
# ==== 3-2 / 3-3. 무할인 반사실 · 정책 비교 ====
# 7-2 무할인 + 7-3 정책 비교
pol_zero = pol_exist.iloc[0:0]
t0=time.time()
sum_z, sal_z, _ = simulate_policy(pol_zero, SIM_START, SIM_END, n_runs=4, elasticity="off",
                                  hour_factor=HOUR_F, cat_factor=CAT_F2, seed0=9500, collect_detail=True)
print(f"7-2 무할인 {time.time()-t0:.0f}s qty={sum_z['qty'].mean():.0f} waste={sum_z['wq'].mean():.0f} profit={sum_z['profit'].mean():,.0f}")
sum_z.to_pickle(OUT/"s_sum_z.pkl"); sal_z.to_pickle(OUT/"s_sal_z.pkl")
print("B1 DONE")

# 7-2 무할인 + 7-3 정책 비교
pol_zero = pol_exist.iloc[0:0]
policies = {"기존정책": pol_exist, "무할인": pol_zero,
            "임박일괄30": uniform_policy(SIM_START, SIM_END, 30, dte_max=1),
            "마감강화2040": closing_boost_policy(SIM_START, SIM_END)}
rows=[]; t0=time.time()
for nm,p in policies.items():
    sm,_,_ = simulate_policy(p, SIM_START, SIM_END, n_runs=2, elasticity="mid",
                             hour_factor=HOUR_F, cat_factor=CAT_F2, seed0=7000)
    rows.append(dict(policy=nm, qty=sm["qty"].mean(), revenue=sm["rev"].mean(), margin=sm["margin"].mean(),
                     waste_qty=sm["wq"].mean(), waste_cost=sm["wc"].mean(),
                     profit=sm["profit"].mean(), profit_sd=sm["profit"].std()))
    print(" ", nm, f"{time.time()-t0:.0f}s")
cmp_pol = pd.DataFrame(rows)
cmp_pol["이익증분_vs기존"] = cmp_pol["profit"] - cmp_pol.loc[cmp_pol.policy=="기존정책","profit"].iloc[0]
cmp_pol.to_pickle(OUT/"s_cmp.pkl")
print(cmp_pol.round(0).to_string(index=False))

In [ ]:
val = res  # 검증 변수 연결 (버그 확정 수정)
# ==== 3-3b / 4. 민감도 + 인터페이스 산출물 ====
# 탄력성 민감도(축소) + 8. 인터페이스 산출물
sens=[]; t0=time.time()
for mode in ["off","mid","upper"]:
    b,_,_ = simulate_policy(closing_boost_policy(SIM_START,SIM_END), SIM_START, SIM_END, n_runs=1,
                            elasticity=mode, hour_factor=HOUR_F, cat_factor=CAT_F2, seed0=8000)
    sens.append(dict(elasticity=mode, k=ELASTICITY[mode], qty=float(b["qty"].iloc[0]),
                     waste_qty=float(b["wq"].iloc[0]), profit=float(b["profit"].iloc[0])))
sens_df = pd.DataFrame(sens); print(f"민감도 {time.time()-t0:.0f}s"); print(sens_df.round(0).to_string(index=False))
IF = OUT/"interface"; NR = sal_ex["run"].nunique(); NZ = sal_z["run"].nunique()
g2 = sal_ex.groupby(["run","store_id","date","hour","product_id"], as_index=False).agg(qty=("qty","sum"), revenue=("revenue","sum"))
full = g2[["store_id","date","hour","product_id"]].drop_duplicates().merge(pd.DataFrame({"run":range(NR)}), how="cross")
(full.merge(g2, how="left").fillna(0)
 .groupby(["store_id","date","hour","product_id"], as_index=False)
 .agg(qty_mean=("qty","mean"), qty_std=("qty","std"), revenue_mean=("revenue","mean"))
 .round(3)).to_csv(IF/"b2_baseline_hourly.csv", index=False, encoding="utf-8-sig")
was_ex.groupby(["store_id","date","product_id"], as_index=False).agg(
    waste_qty=("waste_qty", lambda x: x.sum()/NR), waste_cost=("waste_cost", lambda x: x.sum()/NR)
).round(2).to_csv(IF/"b2_baseline_waste.csv", index=False, encoding="utf-8-sig")
(sal_z.groupby(["store_id","date","hour","product_id"], as_index=False).agg(qty_sum=("qty","sum"))
 .assign(qty_mean=lambda x: x["qty_sum"]/NZ).drop(columns="qty_sum").round(3)
 ).to_csv(IF/"b2_nodiscount_hourly.csv", index=False, encoding="utf-8-sig")
cmp_pol.round(1).to_csv(IF/"b2_policy_comparison_demo.csv", index=False, encoding="utf-8-sig")
sens_df.round(1).to_csv(IF/"b2_elasticity_sensitivity.csv", index=False, encoding="utf-8-sig")
json.dump(dict(hour_factor={str(k):round(v,4) for k,v in HOUR_F.items()},
               cat_factor={k:round(v,4) for k,v in CAT_F2.items()}, global_G=round(G,4),
               ELASTICITY={k:round(v,4) for k,v in ELASTICITY.items()},
               CONSIDER_SIZE=CONSIDER_SIZE, PREF_BOOST=PREF_BOOST, MEM_DECAY=MEM_DECAY),
          open(IF/"b2_calibration.json","w",encoding="utf-8"), ensure_ascii=False, indent=2)
spec = dict(
  policy_schema=["store_id","product_id","date","hour(-1=전시간)","dte_min","dte_max","discount_rate(0~40 정수%)"],
  policy_rule="로트 days_to_expiry가 [dte_min,dte_max]에 들고 hour 일치(-1=전체) 시 적용. 시각지정>전시간, 좁은구간>넓은구간, 동순위면 높은 할인",
  function="simulate_policy(policy_df, start, end, n_runs, elasticity{off|empirical|upper}, hour_factor, cat_factor, collect_detail)",
  visitor_input="visit_counts()가 visitor.csv 실측(store x date x hour)을 반환. A-1 완성 시 이 함수만 A-1 예측(store_id,date,hour,n_visits)으로 교체",
  elasticity_k={k:round(v,4) for k,v in ELASTICITY.items()},
  metrics=["qty","revenue","gross","margin=Σ(판매가-로트원가)","waste_qty","waste_cost","profit=margin-waste_cost"],
  validation={k: round(v,3) if isinstance(v,float) else v for k,v in val.items()},
  b1_dependency="b1_models.joblib 로드 전 ElasticityRuleModel 클래스 정의 필수. B-1 slot 보정 대신 b2_calibration.json의 hour/cat/G 사용")
json.dump(spec, open(IF/"b2_interface_spec.json","w",encoding="utf-8"), ensure_ascii=False, indent=2)
print("산출물:", sorted(p.name for p in IF.iterdir()))

# ==== [신규] A-1 수요 시나리오 3종 (P10/P50/P90) 정책 강건성 데모 ====
# A-1이 안정화되면 SCEN 배율을 실제 분위수 비율(P10/P50, P90/P50)로 교체 (현재는 자리표시 배율)
SCEN = {"P10(저수요)": 0.85, "P50(기준)": 1.00, "P90(고수요)": 1.15}
rows_s = []
for nm, sc in SCEN.items():
    a,_,_ = simulate_policy(pol_exist, SIM_START, SIM_END, n_runs=2, elasticity="mid",
                            hour_factor=HOUR_F, cat_factor=CAT_F2, seed0=8500, anchor_scale=sc)
    rows_s.append(dict(scenario=nm, anchor_scale=sc, qty=a["qty"].mean(),
                       waste_qty=a["wq"].mean(), profit=a["profit"].mean()))
scen_df = pd.DataFrame(rows_s)
print("[A-1 시나리오 강건성 | 기존정책]"); print(scen_df.round(0).to_string(index=False))
scen_df.round(1).to_csv(IF/"b2_a1_scenario_demo.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ==== 5. 결과 요약·다운로드 ====
summary = f"""B-2 실행 요약
검증(기존정책 재현 12/1~14): qty {val['sim_qty']:.0f}/{val['act_qty']} ({val['sim_qty']/val['act_qty']:.3f})
  waste {val['sim_waste']:.0f}/{val['act_waste']} ({val['sim_waste']/val['act_waste']:.3f}) | 시간상관 {val['hour_corr']:.3f} 일상관 {val['day_corr']:.3f}
보정: G={fac_G:.3f} + hour x cat IPF | 수량모델: {QTY_BACKEND}
탄력성 k: {ELASTICITY}
"""
open(OUT/"b2_run_summary.txt","w",encoding="utf-8").write(summary); print(summary)
import shutil; shutil.make_archive("b2_outputs_final","zip",OUT)
try:
    from google.colab import files; files.download("b2_outputs_final.zip")
except Exception as e: print("(로컬 실행: zip은 현재 폴더에 저장됨)", e)